# Day 3: DataFrame Practice Exercises

## Welcome!
These exercises are designed for beginners to practice PySpark DataFrame operations using the COVID-19 dataset. Follow each step carefully to build your skills.

## Before You Start
- Run `docker-compose up` in the `01_basic_spark` directory.
- Open Jupyter at `http://localhost:8888`.
- Place `covid-data.csv` in the `covid-dataset/` directory.
- Download the dataset if needed: https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv

## Exercises



In [1]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("SparkDataFrameExercises").getOrCreate()

---

### Exercise 1: Create a DataFrame from a List
#### What to Do
- Create a DataFrame from `[("AFG", 230375, 0), ("IND", 450000, 50)]` with columns `iso_code`, `total_cases`, and `new_cases`.

#### Steps
- Start a Spark session.
- Use `spark.createDataFrame()` with the list and column names.
- Show the DataFrame using `show()`.


In [2]:
data_1 = data = [("AFG", 230375, 0), ("IND", 450000, 50)]
cols_1 = ["iso_code", "iso_code", "iso_code", ]
df = spark.createDataFrame(data_1, cols_1)

In [3]:
df.show()

+--------+--------+--------+
|iso_code|iso_code|iso_code|
+--------+--------+--------+
|     AFG|  230375|       0|
|     IND|  450000|      50|
+--------+--------+--------+




---

### Exercise 2: Create with a Schema
#### What to Do
- Use the same list as Exercise 1 but define a schema with `iso_code` as string, `total_cases` as integer, and `new_cases` as integer.

#### Steps
- Start a Spark session.
- Define a schema using `StructType` and `StructField`.
- Create the DataFrame with the schema.
- Show the DataFrame and its schema with `printSchema()`.- 

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema_2 = StructType([
    StructField("iso_code", StringType(), True),
    StructField("total_cases", IntegerType(), True),
    StructField("new_cases", IntegerType(), True),
])

In [4]:
#help(spark.createDataFrame)

In [5]:
df = spark.createDataFrame(data_1, schema_2)

---

### Exercise 3: Read CSV and Display
#### What to Do
- Read `covid-data.csv` into a DataFrame and display the first 5 rows.

#### Steps
- Start a Spark session.
- Use `spark.read.csv()` with `header=True` and `inferSchema=True`.
- Show 5 rows with `show(5)` and the schema with `printSchema()`.


In [6]:
df_covid = _df_covid = spark.read.csv("data/owid-covid-data.csv", header=True, inferSchema=True)

In [7]:
df_covid.first()

Row(iso_code='AFG', continent='Asia', location='Afghanistan', date=datetime.date(2020, 1, 5), total_cases=0, new_cases=0, new_cases_smoothed=None, total_deaths=0, new_deaths=0, new_deaths_smoothed=None, total_cases_per_million=0.0, new_cases_per_million=0.0, new_cases_smoothed_per_million=None, total_deaths_per_million=0.0, new_deaths_per_million=0.0, new_deaths_smoothed_per_million=None, reproduction_rate=None, icu_patients=None, icu_patients_per_million=None, hosp_patients=None, hosp_patients_per_million=None, weekly_icu_admissions=None, weekly_icu_admissions_per_million=None, weekly_hosp_admissions=None, weekly_hosp_admissions_per_million=None, total_tests=None, new_tests=None, total_tests_per_thousand=None, new_tests_per_thousand=None, new_tests_smoothed=None, new_tests_smoothed_per_thousand=None, positive_rate=None, tests_per_case=None, tests_units=None, total_vaccinations=None, people_vaccinated=None, people_fully_vaccinated=None, total_boosters=None, new_vaccinations=None, new_v

In [8]:
df_covid.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

---

### Exercise 4: Filter and Aggregate from CSV
#### What to Do
- Filter rows where `new_cases` > 0 and calculate the total `new_cases` for each continent.

#### Steps
- Load the CSV into a DataFrame.
- Use `filter()` to keep rows where `new_cases` > 0.
- Group by `continent` and use `agg()` with `sum("new_cases")`.
- Show the result.



In [8]:
import pyspark.sql.functions as sf

In [9]:
df_covid.count()

429435

In [10]:
df_covid.filter(sf.col("new_cases") > 0).count()

41774

In [11]:
result_df = (
    df_covid
    .filter(sf.col("new_cases") > 0)
    .groupBy("continent")
    .agg(sf.sum("new_cases").alias("total_new_cases"))
)
result_df.show()
#result_df.explain(mode="formatted")

+-------------+---------------+
|    continent|total_new_cases|
+-------------+---------------+
|       Europe|      252916868|
|       Africa|       13146831|
|         NULL|     2512457276|
|North America|      124492698|
|South America|       68811012|
|      Oceania|       15003468|
|         Asia|      301564180|
+-------------+---------------+



---

### Exercise 5: Transform and Calculate KPI
#### What to Do
- Compute the average `total_cases_per_million` for each continent, sorted in descending order.

#### Steps
- Load the CSV into a DataFrame.
- Group by `continent` and use `agg()` with `avg("total_cases_per_million")`.
- Sort using `orderBy()` in descending order.
- Show the result.


In [13]:
result_df = (
    df_covid
    #.filter(sf.col("new_cases") > 0)
    #.filter(col("continent").isNotNull())  # in solutions, but not affecting results
    .groupBy("continent")
    .agg(
        sf.avg("total_cases_per_million")
          .alias("avg_cases_per_million"))
    .orderBy(
        sf.col("avg_cases_per_million")
          .desc()
    )
)
result_df.show()

+-------------+---------------------+
|    continent|avg_cases_per_million|
+-------------+---------------------+
|       Europe|   224006.97074085745|
|North America|   132425.64220692342|
|      Oceania|   113796.86926000625|
|South America|   111028.52515019715|
|         NULL|    95738.93645708887|
|         Asia|    80234.34640331384|
|       Africa|   26604.429582992863|
+-------------+---------------------+



---

### Exercise 6: Count Rows by Year
#### What to Do
- Extract the year from `date` and count the number of records per year.

#### Steps
- Load the CSV into a DataFrame.
- Use `year()` from `pyspark.sql.functions` to extract the year.
- Group by year and count rows.
- Show the result.


In [14]:
result_df = (
    df_covid
    .withColumn("year", sf.year(sf.col("date")))
    .groupBy("year")
    .count()
    .orderBy(sf.col("year"))
)
result_df.show()

+----+-----+
|year|count|
+----+-----+
|2020|90982|
|2021|95088|
|2022|94932|
|2023|93831|
|2024|54602|
+----+-----+



---

### Exercise 7: Filter High Reproduction Rate
#### What to Do
- Filter rows where `reproduction_rate` > 1.5 and show `location`, `date`, and `reproduction_rate`.

#### Steps
- Load the CSV into a DataFrame.
- Use `filter()` to keep rows where `reproduction_rate` > 1.5.
- Select the specified columns.
- Show the first 10 rows.


In [15]:
result_df = (
    df_covid
    .filter(sf.col("reproduction_rate") > 1.5)
    .select(
        sf.col("location"),
        sf.col("date"),
        sf.col("reproduction_rate"),
    ))
result_df.show(10)

+-----------+----------+-----------------+
|   location|      date|reproduction_rate|
+-----------+----------+-----------------+
|Afghanistan|2020-03-29|             1.51|
|Afghanistan|2020-03-30|             1.51|
|Afghanistan|2020-03-31|             1.52|
|Afghanistan|2020-04-01|             1.51|
|Afghanistan|2020-04-02|             1.51|
|Afghanistan|2020-04-23|             1.51|
|Afghanistan|2020-04-24|             1.52|
|Afghanistan|2020-04-25|             1.54|
|Afghanistan|2020-04-26|             1.55|
|Afghanistan|2020-04-27|             1.55|
+-----------+----------+-----------------+
only showing top 10 rows



---

### Exercise 8: Join DataFrames
#### What to Do
- Create two DataFrames: `[("AFG", 230375), ("IND", 450000)]` (columns `iso_code`, `total_cases`) and `[("AFG", "Asia"), ("IND", "Asia")]` (columns `iso_code`, `continent`). Join them on `iso_code`.

#### Steps
- Start a Spark session.
- Create the two DataFrames.
- Use `join()` on `iso_code`.
- Show the result.


In [16]:
df1 = spark.createDataFrame(
    [("AFG", 230375), ("IND", 450000)],
    ["iso_code", "total_cases"],
)
df2 = spark.createDataFrame(
    [("AFG", "Asia"), ("IND", "Asia")],
    ["iso_code", "continent"],
)

In [17]:
df1.join(df2, "iso_code").show()

+--------+-----------+---------+
|iso_code|total_cases|continent|
+--------+-----------+---------+
|     AFG|     230375|     Asia|
|     IND|     450000|     Asia|
+--------+-----------+---------+



---

### Exercise 9: Count Missing Values
#### What to Do
- Count the number of rows where `new_cases` is null.

#### Steps
- Load the CSV into a DataFrame.
- Use `isNull()` to identify missing `new_cases`.
- Count the matching rows.
- Print the count.

In [18]:
(
    df_covid
    .filter(df_covid.new_cases.isNull())
    .count()
)

19276

In [19]:
(
    df_covid
    .filter(sf.col("new_cases").isNull())
    .count()
)

19276

---

### Exercise 10: Add a New Column
#### What to Do
- Add a column `cases_per_population` by dividing `total_cases` by `population`.

#### Steps
- Load the CSV into a DataFrame.
- Use `withColumn()` to create the new column.
- Show 5 rows with `iso_code`, `total_cases`, `population`, and `cases_per_population`.



In [20]:
result_df = (
    df_covid
    .withColumn("cases_per_population", sf.try_divide("total_cases", "population"))
    .filter(sf.col("total_cases") > 0)
    .select(
        sf.col("iso_code"),
        sf.col("total_cases"),
        sf.col("population"),
        sf.col("cases_per_population"),
    )
    #.orderBy(sf.col("cases_per_population").desc())
)
result_df.show(15)

+--------+-----------+----------+--------------------+
|iso_code|total_cases|population|cases_per_population|
+--------+-----------+----------+--------------------+
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          1|  41128772|2.431387934461063E-8|
|     AFG|          7|  41128772|1.701971554122744E-7|
+--------+

---

### Exercise 11: Drop Columns
#### What to Do
- Drop `new_cases_smoothed` and `new_deaths_smoothed` from the DataFrame.

#### Steps
- Load the CSV into a DataFrame.
- Use `drop()` to remove the specified columns.
- Show the first 5 rows.


In [21]:
result_df = (
    df_covid
    .drop(sf.col("new_cases_smoothed"))
    .drop(sf.col("new_deaths_smoothed"))
)
result_df.show(1)

+--------+---------+-----------+----------+-----------+---------+------------+----------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+------------------------------------------+----------------+----------

In [22]:
result_df = (
    df_covid
    .drop(
        sf.col("new_cases_smoothed"),
        sf.col("new_deaths_smoothed"),
        "iso_code",
        "continent",
        df_covid.location,
        df_covid.date,
    )
)
result_df.columns
#result_df.show(1)

['total_cases',
 'new_cases',
 'total_deaths',
 'new_deaths',
 'total_cases_per_million',
 'new_cases_per_million',
 'new_cases_smoothed_per_million',
 'total_deaths_per_million',
 'new_deaths_per_million',
 'new_deaths_smoothed_per_million',
 'reproduction_rate',
 'icu_patients',
 'icu_patients_per_million',
 'hosp_patients',
 'hosp_patients_per_million',
 'weekly_icu_admissions',
 'weekly_icu_admissions_per_million',
 'weekly_hosp_admissions',
 'weekly_hosp_admissions_per_million',
 'total_tests',
 'new_tests',
 'total_tests_per_thousand',
 'new_tests_per_thousand',
 'new_tests_smoothed',
 'new_tests_smoothed_per_thousand',
 'positive_rate',
 'tests_per_case',
 'tests_units',
 'total_vaccinations',
 'people_vaccinated',
 'people_fully_vaccinated',
 'total_boosters',
 'new_vaccinations',
 'new_vaccinations_smoothed',
 'total_vaccinations_per_hundred',
 'people_vaccinated_per_hundred',
 'people_fully_vaccinated_per_hundred',
 'total_boosters_per_hundred',
 'new_vaccinations_smoothed_pe

---

### Exercise 12: Rename Columns
#### What to Do
- Rename `total_cases_per_million` to `cases_per_mil` and `new_cases` to `daily_cases`.

#### Steps
- Load the CSV into a DataFrame.
- Use `withColumnRenamed()` to rename the columns.
- Show the first 5 rows with the renamed columns.



In [23]:
result_df = (
    df_covid
    .withColumnRenamed("total_cases_per_million", "cases_per_mil")
    .withColumnRenamed("new_cases", "daily_cases")
    .select(
        "iso_code",
        "cases_per_mil", "daily_cases",
    )
)
result_df.show(5)

+--------+-------------+-----------+
|iso_code|cases_per_mil|daily_cases|
+--------+-------------+-----------+
|     AFG|          0.0|          0|
|     AFG|          0.0|          0|
|     AFG|          0.0|          0|
|     AFG|          0.0|          0|
|     AFG|          0.0|          0|
+--------+-------------+-----------+
only showing top 5 rows



---

### Exercise 13: Filter by Multiple Conditions
#### What to Do
- Filter rows where `continent` is "Asia" and `new_deaths` > 100, showing `location`, `date`, `new_deaths`.

#### Steps
- Load the CSV into a DataFrame.
- Use `filter()` with `&` for multiple conditions.
- Select the specified columns and show 10 rows.


In [24]:
result_df = (
    df_covid
    .filter(sf.col("continent") == "Asia")
    .filter(sf.col("new_deaths") > 100)
    .filter(sf.col("continent").isNotNull())
    .select(
        "iso_code",
        "continent",
        "date",
        "new_deaths",
        "location",
    )
)
result_df.show(5)

+--------+---------+----------+----------+-----------+
|iso_code|continent|      date|new_deaths|   location|
+--------+---------+----------+----------+-----------+
|     AFG|     Asia|2020-06-14|       124|Afghanistan|
|     AFG|     Asia|2020-06-28|       155|Afghanistan|
|     AFG|     Asia|2020-07-05|       123|Afghanistan|
|     AFG|     Asia|2020-07-12|       149|Afghanistan|
|     AFG|     Asia|2020-07-19|       172|Afghanistan|
+--------+---------+----------+----------+-----------+
only showing top 5 rows



In [25]:
result_df = (
    df_covid
    .filter(
        (sf.col("continent") == "Asia")  # Make sure to wrap conditions in parentheses ()
        & (sf.col("new_deaths") > 100)   # due to bitwise operator precedence before comparison
        & (sf.col("continent").isNotNull())
    )
    .select(
        "iso_code",
        "continent",
        "date",
        "new_deaths",
        "location",
    )
)
result_df.show(5)

+--------+---------+----------+----------+-----------+
|iso_code|continent|      date|new_deaths|   location|
+--------+---------+----------+----------+-----------+
|     AFG|     Asia|2020-06-14|       124|Afghanistan|
|     AFG|     Asia|2020-06-28|       155|Afghanistan|
|     AFG|     Asia|2020-07-05|       123|Afghanistan|
|     AFG|     Asia|2020-07-12|       149|Afghanistan|
|     AFG|     Asia|2020-07-19|       172|Afghanistan|
+--------+---------+----------+----------+-----------+
only showing top 5 rows



In [26]:
result_df = (
    df_covid
    .filter("continent = 'Asia' AND new_deaths > 100 AND continent IS NOT NULL")
    .select(
        "iso_code",
        "continent",
        "date",
        "new_deaths",
        "location",
    )
)
result_df.show(5)

+--------+---------+----------+----------+-----------+
|iso_code|continent|      date|new_deaths|   location|
+--------+---------+----------+----------+-----------+
|     AFG|     Asia|2020-06-14|       124|Afghanistan|
|     AFG|     Asia|2020-06-28|       155|Afghanistan|
|     AFG|     Asia|2020-07-05|       123|Afghanistan|
|     AFG|     Asia|2020-07-12|       149|Afghanistan|
|     AFG|     Asia|2020-07-19|       172|Afghanistan|
+--------+---------+----------+----------+-----------+
only showing top 5 rows



---

### Exercise 14: Count Distinct Locations
#### What to Do
- Count the number of distinct `location`s per `continent`.

#### Steps
- Load the CSV into a DataFrame.
- Group by `continent` and use `agg()` with `countDistinct("location")`.
- Show the result.


In [27]:
result_df = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .groupBy("continent")
    .agg(
        sf.countDistinct("location")
          .alias("distinct_locations"))
    .orderBy(
        sf.col("distinct_locations")
          .desc()
    )
)
result_df.show()

+-------------+------------------+
|    continent|distinct_locations|
+-------------+------------------+
|       Africa|                58|
|       Europe|                55|
|         Asia|                51|
|North America|                41|
|      Oceania|                24|
|South America|                14|
+-------------+------------------+



---

### Exercise 15: Sort by Multiple Columns
#### What to Do
- Sort the DataFrame by `continent` (ascending) and `total_cases` (descending).

#### Steps
- Load the CSV into a DataFrame.
- Use `orderBy()` with `continent` and `total_cases`.
- Show 10 rows with `continent`, `location`, `total_cases`.


In [135]:
#help(df_covid.distinct)

In [28]:
result_df = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    #.distinct()
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
    #.distinct()
    .select(
        "continent",
        "location",
        "total_cases",
    )
    #.distinct()
)
#result_df.explain()
#result_df.explain(mode="formatted")

result_df.show(20)

+---------+------------+-----------+
|continent|    location|total_cases|
+---------+------------+-----------+
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072759|
+---------+------------+-----------+
only showing top 20 rows



### Order of operations matters !

Running `distinct()` before the `orderBy()` will result in 2 shuffle operations 
indicated by `Exchange` in the explain() output.

```
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Filter (2)
                  +- Scan csv  (1)
```

While running `distinct()` after the `orderBy()` will result in only 1 shuffle operations 
indicated by `Exchange` in the explain() output.

```
== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Filter (2)
            +- Scan csv  (1)
```

**But note the caveats explained below !!**

Ref see also: https://stackoverflow.com/questions/56441314/does-distinct-sort-the-dataset

In [29]:
result_df = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .distinct()
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
    .select(
        "continent",
        "location",
        "total_cases",
    )
    
)
#result_df.explain()
result_df.explain(mode="formatted")

#result_df.show(20)

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Filter (2)
                  +- Scan csv  (1)


(1) Scan csv 
Output [67]: [iso_code#42, continent#43, location#44, date#45, total_cases#46, new_cases#47, new_cases_smoothed#48, total_deaths#49, new_deaths#50, new_deaths_smoothed#51, total_cases_per_million#52, new_cases_per_million#53, new_cases_smoothed_per_million#54, total_deaths_per_million#55, new_deaths_per_million#56, new_deaths_smoothed_per_million#57, reproduction_rate#58, icu_patients#59, icu_patients_per_million#60, hosp_patients#61, hosp_patients_per_million#62, weekly_icu_admissions#63, weekly_icu_admissions_per_million#64, weekly_hosp_admissions#65, weekly_hosp_admissions_per_million#66, total_tests#67L, new_tests#68, total_tests_per_thousand#69, new_tests_per_thousand#70, new_tests_smoothed#71, new_tests_smoothed_per_thousand#72, positive_rate#73,

In [30]:
result_df = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
    .distinct()
    .select(
        "continent",
        "location",
        "total_cases",
    )
    
)
#result_df.explain()
result_df.explain(mode="formatted")

#result_df.show(20)

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Filter (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [67]: [iso_code#42, continent#43, location#44, date#45, total_cases#46, new_cases#47, new_cases_smoothed#48, total_deaths#49, new_deaths#50, new_deaths_smoothed#51, total_cases_per_million#52, new_cases_per_million#53, new_cases_smoothed_per_million#54, total_deaths_per_million#55, new_deaths_per_million#56, new_deaths_smoothed_per_million#57, reproduction_rate#58, icu_patients#59, icu_patients_per_million#60, hosp_patients#61, hosp_patients_per_million#62, weekly_icu_admissions#63, weekly_icu_admissions_per_million#64, weekly_hosp_admissions#65, weekly_hosp_admissions_per_million#66, total_tests#67L, new_tests#68, total_tests_per_thousand#69, new_tests_per_thousand#70, new_tests_smoothed#71, new_tests_smoothed_per_thousand#72, positive_rate#73, tests_per_case#74, tests_units#75, total_vaccinations#76L, p

### Additionally sorting order in a .show() output depends on operation order

`df.orderBy().distinct()`

can result in different output order than

`df.distinct().orderBy()`

In [34]:
result_df = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
    .distinct()
    .select(
        "continent",
        "location",
        "total_cases",
    )
)
#result_df.explain()
#result_df.explain(mode="formatted")
print(result_df.count())
result_df.show(20)

402910
+---------+-----------+-----------+
|continent|   location|total_cases|
+---------+-----------+-----------+
|     Asia|Afghanistan|      32672|
|     Asia|Afghanistan|      38919|
|     Asia|Afghanistan|      70761|
|     Asia|Afghanistan|     180123|
|     Asia|Afghanistan|     208097|
|     Asia|Afghanistan|     212612|
|   Europe|    Albania|        186|
|   Europe|    Albania|        539|
|   Europe|    Albania|       7117|
|   Europe|    Albania|     125842|
|   Europe|    Albania|     132484|
|   Europe|    Albania|     161324|
|   Europe|    Albania|     331745|
|   Europe|    Albania|     333883|
|   Europe|    Albania|     334090|
|   Europe|    Albania|     334090|
|   Europe|    Albania|     334896|
|   Europe|    Albania|     335006|
|   Europe|    Albania|     335038|
|   Africa|    Algeria|      38133|
+---------+-----------+-----------+
only showing top 20 rows



In [37]:
result_df = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
    .distinct()
    .select(
        "continent",
        "location",
        "total_cases",
    )
)
result_df.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Filter (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [67]: [iso_code#42, continent#43, location#44, date#45, total_cases#46, new_cases#47, new_cases_smoothed#48, total_deaths#49, new_deaths#50, new_deaths_smoothed#51, total_cases_per_million#52, new_cases_per_million#53, new_cases_smoothed_per_million#54, total_deaths_per_million#55, new_deaths_per_million#56, new_deaths_smoothed_per_million#57, reproduction_rate#58, icu_patients#59, icu_patients_per_million#60, hosp_patients#61, hosp_patients_per_million#62, weekly_icu_admissions#63, weekly_icu_admissions_per_million#64, weekly_hosp_admissions#65, weekly_hosp_admissions_per_million#66, total_tests#67L, new_tests#68, total_tests_per_thousand#69, new_tests_per_thousand#70, new_tests_smoothed#71, new_tests_smoothed_per_thousand#72, positive_rate#73, tests_per_case#74, tests_units#75, total_vaccinations#76L, p

In [32]:
result_df = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .distinct()
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
    .select(
        "continent",
        "location",
        "total_cases",
    )
)
#result_df.explain()
#result_df.explain(mode="formatted")
print(result_df.count())
result_df.show(20)

402910
+---------+------------+-----------+
|continent|    location|total_cases|
+---------+------------+-----------+
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072759|
+---------+------------+-----------+
only showing top 20 rows



In [101]:
result_df = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .select(
        "continent",
        "location",
        "total_cases",
    )
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
    .distinct()
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
)
result_df.show(
    #10
)

+---------+------------+-----------+
|continent|    location|total_cases|
+---------+------------+-----------+
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072751|
|   Africa|South Africa|    4072744|
|   Africa|South Africa|    4072740|
|   Africa|South Africa|    4072726|
|   Africa|South Africa|    4072720|
|   Africa|South Africa|    4072712|
|   Africa|South Africa|    4072708|
|   Africa|South Africa|    4072697|
|   Africa|South Africa|    4072684|
|   Africa|South Africa|    4072681|
|   Africa|South Africa|    4072677|
|   Africa|South Africa|    4072673|
|   Africa|South Africa|    4072661|
|   Africa|South Africa|    4072653|
|   Africa|South Africa|    4072648|
|   Africa|South Africa|    4072642|
|   Africa|South Africa|    4072638|
+---------+------------+-----------+
only showing top 20 rows



---

### Exercise 16: Maximum Cases per Country
#### What to Do
- Find the maximum `total_cases` for each `location`.

#### Steps
- Load the CSV into a DataFrame.
- Group by `location` and use `agg()` with `max("total_cases")`.
- Show the top 5 by maximum cases, sorted descending.

---

### Exercise 17: Rank Locations by Cases
#### What to Do
- Rank locations within each continent by `total_cases` (descending) using a window function.

#### Steps
- Load the CSV into a DataFrame.
- Define a window partitioned by `continent`, ordered by `total_cases` descending.
- Use `rank()` to add a rank column.
- Show 10 rows with `continent`, `location`, `total_cases`, and rank.


---

### Exercise 18: Weekly Average Cases
#### What to Do
- Compute the average `new_cases` per week for each `location` using `weekofyear`.

#### Steps
- Load the CSV into a DataFrame.
- Use `weekofyear()` to extract the week from `date`.
- Group by `location` and week, then use `avg("new_cases")`.
- Show 10 rows.

---

### Exercise 19: Pivot Table by Year
#### What to Do
- Create a pivot table showing total `new_cases` by `continent` and year.

#### Steps
- Load the CSV into a DataFrame.
- Extract the year from `date` using `year()`.
- Use `groupBy("continent").pivot("year").sum("new_cases")`.
- Show the result.



---

### Exercise 20: Save High Cases to CSV
#### What to Do
- Filter rows where `total_cases` > 100000 and save to a CSV file.

#### Steps
- Load the CSV into a DataFrame.
- Filter rows where `total_cases` > 100000.
- Save to `output/high_cases` using `write.csv()` with `header=True`.
- Show 5 rows of the filtered DataFrame.


---

#### Finish Up
- Save as `exercises/03_exercise.ipynb`.
- Verify the file path for `covid-data.csv` (e.g., `covid-dataset/covid-data.csv`).
- Ask your teacher if you need help!